# PATTERN MINING

## IMPORTS

In [ ]:
import pandas as pd
import numpy as np
import os
# For visualizations
import matplotlib.pyplot as plt
import seaborn as sns
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules
from mlxtend.preprocessing import TransactionEncoder

## CLASSES

In [ ]:
class DiabeticDataLoader:
    def __init__(self, df_raw):
        self.df_raw = df_raw.copy()
        self.df_clean = None
        self.df_no_outliers = None
        self.scaler = None

    def clean_data(self):
        df = self.df_raw.copy()

        # Remover colunas constantes
        constant_cols = ['examide', 'citoglipton']
        df.drop(columns=constant_cols, inplace=True, errors='ignore')

        # Substituir '?' por NaN de todas as colunas
        df.replace('?', np.nan, inplace=True)

        # Lidar com max_glu_serum e A1Cresult
        for col in ['max_glu_serum', 'A1Cresult']:
            df[col] = df[col].replace('None', np.nan)
            df[col + '_measured'] = (~df[col].isna()).astype(int)

        # Indicador de peso registrado e conversão
        df['weight_recorded'] = (~df['weight'].isna()).astype(int)
        df['weight'] = df['weight'].astype('category')

        # Colunas categóricas
        categorical_cols = [
            'race', 'gender', 'age', 'admission_type_id', 'discharge_disposition_id',
            'admission_source_id', 'payer_code', 'medical_specialty',
            'diag_1', 'diag_2', 'diag_3', 'max_glu_serum', 'A1Cresult',
            'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride',
            'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone',
            'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide',
            'insulin', 'glyburide-metformin', 'glipizide-metformin',
            'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone',
            'change', 'diabetesMed', 'readmitted'
        ]

        for col in categorical_cols:
            if col in df.columns:
                df[col] = df[col].astype('category')

        self.df_clean = df
        return df

    def remove_outliers(self, df=None):
        if df is None:
            if self.df_clean is None:
                raise ValueError("Data must be cleaned before outlier removal.")
            else:
                df = self.df_clean.copy()

        outlier_cols = [
            'time_in_hospital', 'num_lab_procedures', 'num_medications',
            'number_outpatient', 'number_emergency', 'number_inpatient'
        ]

        self.thresholds = {}
        for col in outlier_cols:
            if col in df.columns:
                # Calcula Min e P99 para ser o thresholds
                low = df[col].min()
                high = df[col].quantile(0.99)
                self.thresholds[col] = (low, high)

        mask = pd.Series(True, index=df.index)
        for feature, (low, high) in self.thresholds.items():
            if feature in df.columns:
                mask &= df[feature].between(low, high)

        df_no_outliers = df.loc[mask].copy()
        self.df_no_outliers = df_no_outliers
        return df_no_outliers

    def get_clean_data(self):
        if self.df_clean is None:
            self.clean_data()
        return self.df_clean

    def get_no_outliers_data(self):
        if self.df_no_outliers is None:
            self.remove_outliers()
        return self.df_no_outliers

    def get_features_target(self):
        df = self.get_no_outliers_data()
        exclude_cols = ['encounter_id', 'patient_nbr', 'readmitted']
        feature_cols = [col for col in df.columns if col not in exclude_cols]
        X = df[feature_cols]
        y = df['readmitted']
        return X, y

    def get_train_test_split(self, test_size=0.2, random_state=42):
        # Limpa os dados de forma global
        df_clean = self.clean_data()

        # Separa X e Y
        X, y = self.get_features_target()

        # Faz o split
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, stratify=y, random_state=random_state
        )

        # Remove outliers no train
        train_combined = pd.concat([X_train, y_train], axis=1)
        train_combined = self.remove_outliers(train_combined)
        X_train = train_combined.drop(columns=['readmitted'])
        y_train = train_combined['readmitted']

        # Padronização
        numeric_cols = [
            'time_in_hospital', 'num_lab_procedures', 'num_medications',
            'number_outpatient', 'number_emergency', 'number_inpatient',
            'num_procedures', 'number_diagnoses'
        ]

        self.scaler = StandardScaler()
        # FIT apenas no treino
        X_train[numeric_cols] = self.scaler.fit_transform(X_train[numeric_cols])
        # TRANSFORM no teste
        X_test[numeric_cols] = self.scaler.transform(X_test[numeric_cols])

        return X_train, X_test, y_train, y_test

## MINING PROCESS

In [ ]:
df_diabetic_data = pd.read_csv(os.path.join('..','data', 'raw','diabetic_data.csv'))
df_diabetic_loader = DiabeticDataLoader(df_diabetic_data)
df_diabetic_clean = df_diabetic_loader.get_clean_data()
df_diabetic_no_outliers = df_diabetic_loader.get_no_outliers_data()

In [15]:
columns_categorical = df_diabetic_no_outliers.astype('category').columns

df_categorical = df_diabetic_no_outliers[columns_categorical].drop(columns=['encounter_id', 'patient_nbr'], errors='ignore')
df_categorical.head()

,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,...,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted,max_glu_serum_measured,A1Cresult_measured,weight_recorded
0,Caucasian,Female,[0-10),NaN,6,25,1,1,NaN,Pediatrics-Endocrinology,...,No,No,No,No,No,No,NO,0,0,0
1,Caucasian,Female,[10-20),NaN,1,1,7,3,NaN,NaN,...,No,No,No,No,Ch,Yes,>30,0,0,0
2,AfricanAmerican,Female,[20-30),NaN,1,1,7,2,NaN,NaN,...,No,No,No,No,No,Yes,NO,0,0,0
3,Caucasian,Male,[30-40),NaN,1,1,7,2,NaN,NaN,...,No,No,No,No,Ch,Yes,NO,0,0,0
4,Caucasian,Male,[40-50),NaN,1,1,7,1,NaN,NaN,...,No,No,No,No,Ch,Yes,NO,0,0,0


In [ ]:
transactions = []
for _, row in df_categorical.iterrows():
    transaction = [f"{col}={val}" for col, val in row.items() if pd.notna(val)]

    transactions.append(transaction)



_IncompleteInputError: incomplete input (1184793233.py, line 7)

In [24]:
transactions[6]

['race=Caucasian',
 'gender=Male',
 'age=[60-70)',
 'admission_type_id=3',
 'discharge_disposition_id=1',
 'admission_source_id=2',
 'time_in_hospital=4',
 'num_lab_procedures=70',
 'num_procedures=1',
 'num_medications=21',
 'number_outpatient=0',
 'number_emergency=0',
 'number_inpatient=0',
 'diag_1=414',
 'diag_2=411',
 'diag_3=V45',
 'number_diagnoses=7',
 'metformin=Steady',
 'repaglinide=No',
 'nateglinide=No',
 'chlorpropamide=No',
 'glimepiride=Steady',
 'acetohexamide=No',
 'glipizide=No',
 'glyburide=No',
 'tolbutamide=No',
 'pioglitazone=No',
 'rosiglitazone=No',
 'acarbose=No',
 'miglitol=No',
 'troglitazone=No',
 'tolazamide=No',
 'insulin=Steady',
 'glyburide-metformin=No',
 'glipizide-metformin=No',
 'glimepiride-pioglitazone=No',
 'metformin-rosiglitazone=No',
 'metformin-pioglitazone=No',
 'change=Ch',
 'diabetesMed=Yes',
 'readmitted=NO',
 'max_glu_serum_measured=0',
 'A1Cresult_measured=0',
 'weight_recorded=0']